In [2]:
from pyIMSRG import *
import numpy as np

emax =3        # maximum number of oscillator quanta in the model space
ref = 'O16'     # reference used for normal ordering
val = ref # valence space

core_generator = 'atan'   # definition of generator eta for decoupling the core (could also use 'white')
smax_core = 10      # limit of integration in flow parameter s for first stage of decoupling
#smax_core = 0       # limit of integration in flow parameter s for first stage of decoupling

##### Example format of how to read input interaction matrix elements from file (these are not included with the code)
#f2b='input/TwBME-HO_NN-only_N3LO_EM500_srg1.8_hw16_emax14_e2max28.me2j.gz'
#f2e1,f2e2,f2l = 14,28,14
#f3b='input/NO2B_ThBME_EM1.8_2.0_3NFJmax15_IS_hw16_ms18_36_18.stream.bin'
#f3e1,f3e2,f3e3 = 18,36,18
#mode3n='no2b'
#LECs = 'EM1820'
#hw=16

#### Example format of how to read input interaction matrix elements from file (these are not included with the code)
f2b='input/TwBME-HO_NN-only_N3LO_EM500_srg1.8_hw16_emax14_e2max28.me2j.gz'
f2e1,f2e2,f2l = 14,28,14
f3b='input/NO2B_ThBME_EM7.5_1.8_2.0_IS_hw16from16_ms14_28_18.me3j.gz'
f3e1,f3e2,f3e3 = 14,28,18
mode3n='no2b'
LECs = 'EM7.5_1820'
hw=16



#hw = 20    # harmonic oscillator basis frequency
#LECs='Minnesota'
#f3b='none'

##########################################################################
###  END PARAMETER SETTING. BEGIN ACTUALLY DOING STUFF ##################
##########################################################################


### Create an instance of the ModelSpace class
ms = ModelSpace(emax,ref,val)
ms.SetHbarOmega(hw)

### the ReadWrite object handles reading and writing of files
rw = ReadWrite()

rank_j, parity, rank_Tz, particle_rank = 0,0,0,2
if f3b != 'none':
   particle_rank = 3

### Create an instance of the Operator class, representing the Hamiltonian
H = Operator(ms,rank_j, parity, rank_Tz, particle_rank)

### Either generate the matrix elements of the Minnesota potential, or read in matrix elements from file
if LECs == 'Minnesota':
    H += OperatorFromString(ms,'VMinnesota')

else:
  ### Read Two-body matrix elements
  rw.ReadBareTBME_Darmstadt(f2b,H,f2e1,f2e2,f2l)
  ### Read Three-body matrix elements
  if f3b != 'none':
     if mode3n == 'no2b':
        H.ThreeBody.SetMode('no2b')
        H.ThreeBody.ReadFile([f3b],[f3e1,f3e2,f3e3])
     else:
        rw.Read_Darmstadt_3body(f3b,H,f3e1,f3e2,f3e3)


### Add the relative kinetic energy, so H = Trel + V
H += OperatorFromString(ms,'Trel')
print('after reading files, 3-body norm is',H.ThreeBodyNorm())

### Create an instance of the HartreeFock class, used for solving the Hartree-Fock equations
hf = HartreeFock(H)
hf.Solve()
hf.PrintSPEandWF()

### Do normal ordering with respect to the HF basis, and retain only up to 2-body operators
HNO = hf.GetNormalOrderedH(2)

### Create an instance of the IMSRGSolver class, used for solving the IMSRG flow equations
imsrgsolver = IMSRGSolver(HNO)
imsrgsolver.SetMethod('magnus')  # Solve using the Magnus formulation. Could also be 'flow_RK4'

imsrgsolver.SetGenerator(core_generator)
imsrgsolver.SetSmax(smax_core)

### Do the first stage of integration to decouple the core
imsrgsolver.Solve()


### Hs is the IMSRG-evolved Hamiltonian
Hs = imsrgsolver.GetH_s()



Read 5696 matrix elements 
after reading files, 3-body norm is 130.51175702074318
ReadFile. from the input, I extracted input/NO2B_ThBME_EM7.5_1.8_2.0_IS_hw16from16_ms14_28_18.me3j.gz  14 28 18 14
ReadFile  reading/storing with 32  bit floats. filemode is gz
Allocated a vector of size 4312200
Done reading
Calculating moshinsky with Lmax = 3
done calculating moshinsky (389 elements)
Hash table has 397 buckets and a load factor 0.979849  estimated storage ~ 1.17123e-05 GB
HartreeFock::BuildMonopoleV3  storing 26624 doubles for Vmon3 and 26624 uint64's for Vmon3_keys.
HF converged after 25 iterations. 
e1hf = 229.3887610
e2hf = -340.9966615
e3hf = 38.5090244
EHF = -73.0988761
  i:   n   l  2j 2tz            SPE         occ.   |    overlaps
  0:   0   0   1  -1     -35.740064     1.000000   |  0.992472  -0.122475  
  1:   0   0   1   1     -39.226856     1.000000   |  0.993625  -0.112733  
  2:   0   1   3  -1     -15.970211     1.000000   |  0.993073  -0.117502  
  3:   0   1   3   1     

In [3]:
def Norm3(T1, T2):
    T1dag = gm.Get_adjoint(T1)
    T2dag = gm.Get_adjoint(T2)
    
    s_plus =  T1 + T1dag
    s_plus.SetHermitian()
    
    s_minus = T2 - T2dag
    s_minus.SetAntiHermitian()
    
    #s_plus.PrintOneBody()
    #s_minus.PrintOneBody()
    ht_full= HNO*0
    ht_full.SetHermitian()
    
    cm.comm110ss(s_plus,s_minus, ht_full)
    norm_all1= ht_full.ZeroBody/2
    
    ht_full= HNO*0
    ht_full.SetHermitian()
    cm.comm220ss(s_plus,s_minus, ht_full)
    norm_all2= ht_full.ZeroBody/2
    return(norm_all2,norm_all1)

In [15]:
cm=Commutator
gm=Generator()

chi=gm.GetHod_CC(H,"left")
#chi.PrintOneBody()
#chi.EraseTwoBody()

#chi.PrintTwoBody_ch(6)
#ms.Print()
print(Norm(chi,chi))
print(Norm2(chi,chi))
print(Norm3(chi,chi))


3148.1017787128258
3148.101778712825
(1123.1017787128246, 2025.0)


In [5]:
## initialize the T and D^dagger

def htc(Haml, TT):
    
    Tdag = gm.Get_adjoint(TT)
    s_plus =  TT + Tdag
    s_plus.SetHermitian()
    
    s_minus = TT - Tdag
    s_minus.SetAntiHermitian()
    
    ht_plus= Haml*0
    ht_minus= Haml*0
    
    ht_plus.SetAntiHermitian()
    
    ht_minus.SetHermitian()
    
    ht_plus = cm.Commutator(Haml, s_plus )
    
    ht_minus = cm.Commutator(Haml, s_minus )
    
    ht_full = HNO*0
    ht_full.SetNonHermitian()
    ht_full = (ht_plus + ht_minus)/2
    #ht_full.PrintOneBody()
    ht_od = gm.GetHod_CC(ht_full,"left")
    #ht_od.PrintOneBody()
    return(ht_od)


def Norm2(T1, T2):
    T1dag = gm.Get_adjoint(T1)
    T2dag = gm.Get_adjoint(T2)
    
    s_plus =  T1 + T1dag
    s_plus.SetHermitian()
    
    s_minus = T2 - T2dag
    s_minus.SetAntiHermitian()
    
    ht_full= HNO*0
    ht_full.SetHermitian()
    ht_full=cm.Commutator(s_plus,s_minus)
    norm_all= ht_full.ZeroBody/2
    return(norm_all)

def Norm(T1, T2):
    return(gm.GetOverlap(T1,T2))

import numpy as np

def lanczos_proc( hv_func, norm_func, haml, vi, ndim):
    lanczos_vector = []
    hall = np.zeros([ndim,ndim])
    hall[0,0]=0.

    ## normalize it to 1
    nn=norm_func(vi,vi)

    vi=vi/np.sqrt(nn)
    lanczos_vector.append(vi)

    for j in range(ndim):
        
        w = hv_func(haml,lanczos_vector[j])
    
        ai=norm_func(w,lanczos_vector[j])
        #print(j,Norm2(w,w), norm_func(w,w),ai)
    
        if(j>0):
            w=w-ai*lanczos_vector[j]-bj*lanczos_vector[j-1]
        else:
            w=w-ai*lanczos_vector[j]
        
        hall[j,j]=ai


        bj = np.sqrt(norm_func(w,w))
        #print(j,Norm(w,w), norm_func(w,w))

        if bj < 0.0 :
            break
        lanczos_vector.append(w/bj)
    
        if(j<ndim-1):
            hall[j,j+1]=bj
            hall[j+1,j]=bj
        #print(j,ai,bj)
    #print(hall)
    e,v = np.linalg.eig(hall)
    
    return(e,v,lanczos_vector)


In [6]:
chi=gm.GetHod_CC(H,"left")
#chi.PrintOneBody()
#hnew = htc(Hs, chi)
ndim=20
e,v,lvs = lanczos_proc( htc, Norm, Hs, chi, ndim)
e=np.sort(e)
print(e)

[ 10.47056641  22.75074388  27.61488627  29.67108081  35.90224298
  40.89632853  47.58177941  53.1307846   62.89416029  70.12023167
  79.98513615  87.28239398  94.84392362 102.68321944 109.29566749
 114.2148197  120.46928818 131.46134162 132.61961416 134.91018172]


In [19]:
rank_j, parity, rank_Tz, particle_rank = 0,0,0,2
rk = Operator(ms,rank_j, parity, rank_Tz, particle_rank)

In [20]:
rk.PrintTwoBody_ch(1)

0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000


In [21]:
hrk=cm.Commutator(Hs,rk)